# Cross-Model Analysis

Consolidates the four headline results across all models with complete representations
(Llama-3.1-8B, Qwen2.5-7B, Gemma-2-9B). Each figure has per-model panels and collapses
the framework dimension into a mean ± 1σ band; per-framework splits live in single-model
notebooks (08, 09, 10).

| Figure | Question | Panels |
|--------|----------|--------|
| 1 | Does the harmful/benign direction displace with accumulated context? | 1 × N_models |
| 2 | Does displacement break single-turn detection, and can an MLP recover it? | bars: N_models × 3 |
| 3 | Do Zhao et al.'s t_inst/t_post positional roles survive multi-turn? | 2 × N_models |
| 4 | Is the harmfulness signal in the messages or the conversation history? | 2 × N_models |

Models without extracted representations are silently skipped — the notebook runs
fine with only Llama present.

## 1. Setup

In [ ]:
import json
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

repo_root = Path('..').resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

MODELS            = ['llama', 'qwen', 'gemma']
FRAMEWORKS        = ['crescendo', 'actorattack', 'xteaming']
SPLITS            = ['harmful', 'benign']
TRAIN_MAX_ATTEMPT = 16
MAX_K             = 10
H_KEY_DEFAULT     = 'h_inst'
RNG_SEED          = 42

MODEL_LABELS = {'llama': 'Llama-3.1-8B', 'qwen': 'Qwen2.5-7B', 'gemma': 'Gemma-2-9B'}
MODEL_COLORS = {'llama': '#1f77b4', 'qwen': '#ff7f0e', 'gemma': '#2ca02c'}

FIG_DIR = repo_root / 'figures'
FIG_DIR.mkdir(exist_ok=True)
plt.rcParams.update({'figure.dpi': 150, 'savefig.dpi': 200, 'savefig.bbox': 'tight'})


def load_layer_info(model_root):
    for sub in ('trajectories', 'nocontext', 'single_turn'):
        root = model_root / sub
        if not root.exists():
            continue
        for d in root.iterdir():
            fp = d / 'layer_indices.json'
            if fp.exists():
                return json.loads(fp.read_text())
    return None


def cosine(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12))


## 2. Loading

Per model: load the four extraction conditions if present. Models without any
representations are skipped with a warning.

In [ ]:
def load_repr(folder):
    meta   = pd.read_parquet(folder / 'metadata.parquet')
    h_inst = np.load(folder / 'h_inst.npy')
    h_post = np.load(folder / 'h_post_inst.npy')
    assert len(meta) == len(h_inst) == len(h_post)
    return {'meta': meta, 'h_inst': h_inst, 'h_post': h_post}


def add_split(meta):
    if 'attempt' in meta.columns:
        meta['data_split'] = np.where(meta['attempt'] <= TRAIN_MAX_ATTEMPT, 'train', 'test')
    return meta


def load_model(name):
    repr_root  = repo_root / 'data' / name / 'representations'
    layer_info = load_layer_info(repr_root)
    if layer_info is None:
        print(f'  [skip] {name}: no layer_indices.json under {repr_root}')
        return None
    out = {'name': name, 'repr_root': repr_root, 'layer_info': layer_info,
           'traj': {}, 'nc': {}, 'comp': {}, 'st': {}}
    for fw in FRAMEWORKS:
        for split in SPLITS:
            for key, sub in [('traj', 'trajectories'), ('nc', 'nocontext'), ('comp', 'compressed')]:
                folder = repr_root / sub / f'{fw}_{split}'
                if folder.exists():
                    d = load_repr(folder)
                    add_split(d['meta'])
                    out[key][(fw, split)] = d
    for split in SPLITS:
        folder = repr_root / 'single_turn' / split
        if folder.exists():
            out['st'][split] = load_repr(folder)
    n_traj = sum(len(v['meta']) for v in out['traj'].values())
    n_nc   = sum(len(v['meta']) for v in out['nc'].values())
    n_comp = sum(len(v['meta']) for v in out['comp'].values())
    n_st   = sum(len(v['meta']) for v in out['st'].values())
    out['focal_pos']   = layer_info['n_sweep'] // 2     # proportional mid-depth position (4 of 8)
    out['focal_label'] = layer_info['labels'][out['focal_pos']]
    print(f"  {name:8s}: traj={n_traj:6d}  nc={n_nc:6d}  comp={n_comp:6d}  st={n_st:4d}  "
          f"focal={out['focal_label']}  layers={layer_info['labels']}")
    return out


MD = {}
for m in MODELS:
    d = load_model(m)
    if d is not None:
        MD[m] = d
print(f"\nLoaded {len(MD)} model(s): {list(MD.keys())}")


## 3. Directions

For each loaded model, compute:
- `v_st[h_key]` — single-turn anchor, paired subtraction on the 100 JBB pairs.
- `v_full[(fw, h_key)][k]` — per-turn full-context direction.
- `v_nc[(fw, h_key)][k]` — per-turn no-context direction.
- `v_comp[(fw, h_key)]` — one direction per framework from the compressed condition.

All directions are `(n_saved_layers, D)` unit vectors.

In [ ]:
def compute_direction_paired(h_harm, m_harm, h_beni, m_beni):
    pairs = sorted(set(m_harm['pair_id']) & set(m_beni['pair_id']))
    diffs = []
    for pid in pairs:
        idx_h = m_harm.index[m_harm['pair_id'] == pid].tolist()
        idx_b = m_beni.index[m_beni['pair_id'] == pid].tolist()
        if not idx_h or not idx_b:
            continue
        mu_h = h_harm[idx_h].astype(np.float32).mean(0)
        mu_b = h_beni[idx_b].astype(np.float32).mean(0)
        diffs.append(mu_h - mu_b)
    if not diffs:
        return None
    d = np.stack(diffs).mean(0)
    return d / (np.linalg.norm(d, axis=-1, keepdims=True) + 1e-12)


def direction_at_k(data, fw, h_key, k):
    d_h = data.get((fw, 'harmful'))
    d_b = data.get((fw, 'benign'))
    if d_h is None or d_b is None:
        return None
    mh = (d_h['meta']['turn_k'] == k) & (d_h['meta']['data_split'] == 'train')
    mb = (d_b['meta']['turn_k'] == k) & (d_b['meta']['data_split'] == 'train')
    if mh.sum() == 0 or mb.sum() == 0:
        return None
    return compute_direction_paired(
        d_h[h_key][mh.values], d_h['meta'][mh].reset_index(drop=True),
        d_b[h_key][mb.values], d_b['meta'][mb].reset_index(drop=True),
    )


def direction_overall(data, fw, h_key):
    d_h = data.get((fw, 'harmful'))
    d_b = data.get((fw, 'benign'))
    if d_h is None or d_b is None:
        return None
    mh = d_h['meta']['data_split'] == 'train'
    mb = d_b['meta']['data_split'] == 'train'
    return compute_direction_paired(
        d_h[h_key][mh.values], d_h['meta'][mh].reset_index(drop=True),
        d_b[h_key][mb.values], d_b['meta'][mb].reset_index(drop=True),
    )


def direction_st(st, h_key):
    if 'harmful' not in st or 'benign' not in st:
        return None
    m_h = st['harmful']['meta'].assign(_src_idx=lambda x: x.index).sort_values('pair_id')
    m_b = st['benign']['meta'].assign(_src_idx=lambda x: x.index).sort_values('pair_id')
    common = sorted(set(m_h['pair_id']) & set(m_b['pair_id']))
    idx_h = [m_h.loc[m_h['pair_id'] == p, '_src_idx'].iloc[0] for p in common]
    idx_b = [m_b.loc[m_b['pair_id'] == p, '_src_idx'].iloc[0] for p in common]
    diffs = st['harmful'][h_key][idx_h].astype(np.float32) - st['benign'][h_key][idx_b].astype(np.float32)
    d = diffs.mean(0)
    return d / (np.linalg.norm(d, axis=-1, keepdims=True) + 1e-12)


for m, d in MD.items():
    d['v_st'] = {hk: direction_st(d['st'], hk) for hk in ['h_inst', 'h_post']}
    d['v_full'], d['v_nc'], d['v_comp'] = {}, {}, {}
    for fw in FRAMEWORKS:
        for hk in ['h_inst', 'h_post']:
            d['v_full'][(fw, hk)] = {k: direction_at_k(d['traj'], fw, hk, k) for k in range(1, MAX_K + 1)}
            d['v_nc'][(fw, hk)]   = {k: direction_at_k(d['nc'],   fw, hk, k) for k in range(1, MAX_K + 1)}
            d['v_comp'][(fw, hk)] = direction_overall(d['comp'], fw, hk)
    print(f'  {m}: directions computed')


## 4. Figure 1 — Direction displacement across turns

Per model, at the proportional mid-depth layer:

- `cos(v_full(k), v_ST)` — how far the multi-turn direction has rotated from the single-turn anchor.
- `cos(v_nc(k), v_ST)` — same for no-context (isolates pure effect of accumulated history).
- compressed — horizontal reference (Bullwinkel et al. comparison).

Frameworks collapsed into mean ± 1σ band.

In [ ]:
H_KEY = H_KEY_DEFAULT
n = max(1, len(MD))
fig, axes = plt.subplots(1, n, figsize=(4.3 * n, 3.6), sharey=True, squeeze=False)
axes = axes[0]

for ax, (m, d) in zip(axes, MD.items()):
    L = d['focal_pos']
    v_st = d['v_st'][H_KEY]
    if v_st is None:
        ax.text(0.5, 0.5, 'no v_ST', ha='center', va='center', transform=ax.transAxes)
        continue
    v_st_L = v_st[L]
    ks = np.arange(1, MAX_K + 1)
    full_cos = np.full((len(FRAMEWORKS), MAX_K), np.nan)
    nc_cos   = np.full((len(FRAMEWORKS), MAX_K), np.nan)
    comp_vals = []
    for i, fw in enumerate(FRAMEWORKS):
        for j, k in enumerate(ks):
            vf = d['v_full'].get((fw, H_KEY), {}).get(k)
            if vf is not None:
                full_cos[i, j] = cosine(vf[L], v_st_L)
            vnc = d['v_nc'].get((fw, H_KEY), {}).get(k)
            if vnc is not None:
                nc_cos[i, j] = cosine(vnc[L], v_st_L)
        vc = d['v_comp'].get((fw, H_KEY))
        if vc is not None:
            comp_vals.append(cosine(vc[L], v_st_L))

    with warnings.catch_warnings():
        warnings.simplefilter('ignore', category=RuntimeWarning)
        fm, fs = np.nanmean(full_cos, 0), np.nanstd(full_cos, 0)
        nm, ns = np.nanmean(nc_cos, 0),   np.nanstd(nc_cos, 0)

    ax.plot(ks, fm, 'o-', color='#1f77b4', lw=2, label='full-context')
    ax.fill_between(ks, fm - fs, fm + fs, alpha=0.15, color='#1f77b4')
    ax.plot(ks, nm, 's-', color='#d62728', lw=2, label='no-context')
    ax.fill_between(ks, nm - ns, nm + ns, alpha=0.15, color='#d62728')
    if comp_vals:
        ax.axhline(float(np.mean(comp_vals)), color='#2ca02c', ls=':', lw=1.8,
                   label=f'compressed (μ={np.mean(comp_vals):.2f})')
    ax.axhline(0, color='gray', lw=0.5)
    ax.set_xticks(ks)
    ax.set_xlabel('turn k')
    ax.set_ylim(-0.15, 1.05)
    ax.grid(alpha=0.3)
    ax.set_title(f"{MODEL_LABELS[m]}  ({d['focal_label']})")
    ax.legend(fontsize=8, loc='lower left', frameon=False)

axes[0].set_ylabel(r'$\cos(v_{\mathrm{harmful}}(k),\,v_{\mathrm{ST}})$')
fig.suptitle('Fig 1 — Direction displacement across turns (frameworks averaged)', y=1.03)
fig.tight_layout()
fig.savefig(FIG_DIR / '14_fig1_displacement.png')
plt.show()


## 5. Figure 2 — Detection failure & linearity

Three bars per model, averaged over mid-turns k ∈ {3, 4, 5} on the test split at the
focal layer:

1. **fixed-ST linear** — project test activations onto `v_ST` (single-turn direction).
2. **adaptive linear** — project onto `v_full(k)` (per-turn direction trained on train split).
3. **ST → full MLP** — MLP trained on single-turn activations, evaluated on multi-turn test.

If bar 2 ≫ bar 1, displacement breaks fixed defenses. If bar 3 ≈ bar 1, an MLP can't recover
the lost signal — displacement is structural, not a linear rotation.

In [ ]:
MID_KS = [3, 4, 5]
H_KEY = H_KEY_DEFAULT


def fixed_st_auroc(model_d, fw):
    L = model_d['focal_pos']
    v_st = model_d['v_st'][H_KEY]
    if v_st is None:
        return np.nan
    aucs = []
    for k in MID_KS:
        Xh = model_d['traj'].get((fw, 'harmful'));  Xb = model_d['traj'].get((fw, 'benign'))
        if Xh is None or Xb is None:
            continue
        mh = (Xh['meta']['turn_k'] == k) & (Xh['meta']['data_split'] == 'test')
        mb = (Xb['meta']['turn_k'] == k) & (Xb['meta']['data_split'] == 'test')
        if mh.sum() == 0 or mb.sum() == 0:
            continue
        h = np.concatenate([Xh[H_KEY][mh.values, L, :], Xb[H_KEY][mb.values, L, :]]).astype(np.float32)
        y = np.concatenate([np.ones(mh.sum()), np.zeros(mb.sum())])
        aucs.append(roc_auc_score(y, h @ v_st[L]))
    return float(np.mean(aucs)) if aucs else np.nan


def adaptive_linear_auroc(model_d, fw):
    L = model_d['focal_pos']
    aucs = []
    for k in MID_KS:
        v = model_d['v_full'].get((fw, H_KEY), {}).get(k)
        if v is None:
            continue
        Xh = model_d['traj'].get((fw, 'harmful'));  Xb = model_d['traj'].get((fw, 'benign'))
        mh = (Xh['meta']['turn_k'] == k) & (Xh['meta']['data_split'] == 'test')
        mb = (Xb['meta']['turn_k'] == k) & (Xb['meta']['data_split'] == 'test')
        if mh.sum() == 0 or mb.sum() == 0:
            continue
        h = np.concatenate([Xh[H_KEY][mh.values, L, :], Xb[H_KEY][mb.values, L, :]]).astype(np.float32)
        y = np.concatenate([np.ones(mh.sum()), np.zeros(mb.sum())])
        aucs.append(roc_auc_score(y, h @ v[L]))
    return float(np.mean(aucs)) if aucs else np.nan


def st_to_full_mlp_auroc(model_d, fw):
    L = model_d['focal_pos']
    if 'harmful' not in model_d['st'] or 'benign' not in model_d['st']:
        return np.nan
    Xh_st = model_d['st']['harmful'][H_KEY][:, L, :].astype(np.float32)
    Xb_st = model_d['st']['benign'][H_KEY][:, L, :].astype(np.float32)
    X_tr  = np.concatenate([Xh_st, Xb_st])
    y_tr  = np.concatenate([np.ones(len(Xh_st)), np.zeros(len(Xb_st))])

    clf = make_pipeline(
        StandardScaler(),
        MLPClassifier(hidden_layer_sizes=(64,), max_iter=400, early_stopping=True,
                      random_state=RNG_SEED),
    )
    clf.fit(X_tr, y_tr)

    aucs = []
    for k in MID_KS:
        Xh = model_d['traj'].get((fw, 'harmful'));  Xb = model_d['traj'].get((fw, 'benign'))
        if Xh is None or Xb is None:
            continue
        mh = (Xh['meta']['turn_k'] == k) & (Xh['meta']['data_split'] == 'test')
        mb = (Xb['meta']['turn_k'] == k) & (Xb['meta']['data_split'] == 'test')
        if mh.sum() == 0 or mb.sum() == 0:
            continue
        X_te = np.concatenate([Xh[H_KEY][mh.values, L, :], Xb[H_KEY][mb.values, L, :]]).astype(np.float32)
        y_te = np.concatenate([np.ones(mh.sum()), np.zeros(mb.sum())])
        aucs.append(roc_auc_score(y_te, clf.predict_proba(X_te)[:, 1]))
    return float(np.mean(aucs)) if aucs else np.nan


rows = []
for m, d in MD.items():
    for fw in FRAMEWORKS:
        rows.append({
            'model':     m,
            'framework': fw,
            'fixed_ST':    fixed_st_auroc(d, fw),
            'adaptive':    adaptive_linear_auroc(d, fw),
            'ST_to_MLP':   st_to_full_mlp_auroc(d, fw),
        })
df2 = pd.DataFrame(rows)
df2


In [ ]:
agg = df2.groupby('model')[['fixed_ST', 'adaptive', 'ST_to_MLP']].mean().loc[list(MD)]
fig, ax = plt.subplots(figsize=(max(5, 2.2 * len(MD)), 3.5))

bar_labels = {
    'fixed_ST':  'fixed-ST linear',
    'adaptive':  'adaptive linear (per-turn)',
    'ST_to_MLP': 'ST → full MLP',
}
bar_colors = {'fixed_ST': '#d62728', 'adaptive': '#1f77b4', 'ST_to_MLP': '#9467bd'}

x = np.arange(len(agg))
w = 0.25
for i, col in enumerate(['fixed_ST', 'adaptive', 'ST_to_MLP']):
    ax.bar(x + (i - 1) * w, agg[col].values, width=w,
           label=bar_labels[col], color=bar_colors[col])

ax.set_xticks(x)
ax.set_xticklabels([MODEL_LABELS[m] for m in agg.index])
ax.set_ylabel('AUROC  (mean over k∈3..5, frameworks)')
ax.set_ylim(0.45, 1.02)
ax.axhline(0.5, color='gray', ls=':', lw=0.8)
ax.grid(axis='y', alpha=0.3)
ax.legend(frameon=False, loc='lower right', fontsize=9)
ax.set_title('Fig 2 — Detection failure and linearity (focal layer, test split)')
fig.tight_layout()
fig.savefig(FIG_DIR / '14_fig2_detection.png')
plt.show()

print('\nPer-model means (averaged across frameworks):')
print(agg.round(3))


## 6. Figure 3 — Position collapse (t_inst vs t_post)

Zhao et al. (single-turn): `t_inst` encodes *harmfulness*, `t_post` encodes *refusal*.

Test whether that separation survives multi-turn.

- **Harmfulness task (top row):** per-turn direction `v_full(k)` at each position, AUROC on test
  harmful-vs-benign at the final turn of each conversation.
- **Refusal task (bottom row):** within harmful conversations only, AUROC for
  refused-vs-accepted. Uses conversation-level `attack_success` as the label proxy
  (per-turn `judge_success` is only filled for Crescendo).

Frameworks averaged.

In [ ]:
def final_turn_idx(meta):
    return meta.groupby('conversation_id')['turn_k'].idxmax().values


def task_auroc_direction(d, fw, h_key, pos, task):
    '''AUROC of projection onto v_full(final-turn direction) at every saved layer.
       task ∈ {'harmfulness', 'refusal'}. Returns array shape (n_saved,).'''
    Xh = d['traj'].get((fw, 'harmful'))
    Xb = d['traj'].get((fw, 'benign'))
    if Xh is None:
        return None
    n_saved = d['layer_info']['n_sweep']

    if task == 'harmfulness':
        if Xb is None:
            return None
        # Train direction on train split of harmful + benign (ignore turn_k — use final turns for eval)
        # For this cross-model plot we reuse the overall paired direction at each layer.
        v = direction_overall(d['traj'], fw, h_key)
        if v is None:
            return None
        idx_h = final_turn_idx(Xh['meta']);  idx_b = final_turn_idx(Xb['meta'])
        mh_te = (Xh['meta'].loc[idx_h, 'data_split'] == 'test').values
        mb_te = (Xb['meta'].loc[idx_b, 'data_split'] == 'test').values
        if mh_te.sum() == 0 or mb_te.sum() == 0:
            return None
        Hh = Xh[h_key if pos == 't_inst' else 'h_post'][idx_h][mh_te].astype(np.float32)
        Hb = Xb[h_key if pos == 't_inst' else 'h_post'][idx_b][mb_te].astype(np.float32)
        y  = np.concatenate([np.ones(len(Hh)), np.zeros(len(Hb))])
        out = np.full(n_saved, np.nan)
        for L in range(n_saved):
            proj = np.concatenate([Hh[:, L, :] @ v[L], Hb[:, L, :] @ v[L]])
            out[L] = roc_auc_score(y, proj)
        return out

    if task == 'refusal':
        # Within harmful only: refused (attack_success == False) vs accepted.
        if 'attack_success' not in Xh['meta'].columns:
            return None
        idx_h  = final_turn_idx(Xh['meta'])
        meta_h = Xh['meta'].loc[idx_h]
        test_mask = (meta_h['data_split'] == 'test').values
        idx_h  = idx_h[test_mask]
        meta_h = meta_h[test_mask]
        y_ref  = (~meta_h['attack_success'].astype(bool)).astype(int).values    # 1 = refused
        if len(np.unique(y_ref)) < 2:
            return None
        # Centroid direction within harmful: mean(refused) − mean(accepted) on train split
        tr_idx  = final_turn_idx(Xh['meta'])
        tr_meta = Xh['meta'].loc[tr_idx]
        tr_mask = (tr_meta['data_split'] == 'train').values
        tr_idx  = tr_idx[tr_mask];  tr_meta = tr_meta[tr_mask]
        if len(np.unique(tr_meta['attack_success'])) < 2:
            return None
        arr = Xh[h_key if pos == 't_inst' else 'h_post'][tr_idx].astype(np.float32)
        y_tr = (~tr_meta['attack_success'].astype(bool)).astype(int).values
        mu_r = arr[y_tr == 1].mean(0);  mu_a = arr[y_tr == 0].mean(0)
        v = mu_r - mu_a
        v = v / (np.linalg.norm(v, axis=-1, keepdims=True) + 1e-12)
        X_te = Xh[h_key if pos == 't_inst' else 'h_post'][idx_h].astype(np.float32)
        out = np.full(n_saved, np.nan)
        for L in range(n_saved):
            out[L] = roc_auc_score(y_ref, X_te[:, L, :] @ v[L])
        return out
    return None


# Aggregate across frameworks per (model, task, position)
fig, axes = plt.subplots(2, max(1, len(MD)), figsize=(3.8 * max(1, len(MD)), 6), sharey=True, squeeze=False)
tasks = ['harmfulness', 'refusal']

for row, task in enumerate(tasks):
    for col, (m, d) in enumerate(MD.items()):
        ax = axes[row, col]
        n_saved = d['layer_info']['n_sweep']
        labels  = d['layer_info']['labels']
        for pos, color, marker in [('t_inst', '#1f77b4', 'o'), ('t_post', '#ff7f0e', 's')]:
            per_fw = np.full((len(FRAMEWORKS), n_saved), np.nan)
            for i, fw in enumerate(FRAMEWORKS):
                r = task_auroc_direction(d, fw, 'h_inst', pos, task)
                if r is not None:
                    per_fw[i] = r
            with warnings.catch_warnings():
                warnings.simplefilter('ignore', category=RuntimeWarning)
                mu, sd = np.nanmean(per_fw, 0), np.nanstd(per_fw, 0)
            ax.plot(range(n_saved), mu, marker=marker, color=color, lw=1.8, label=pos)
            ax.fill_between(range(n_saved), mu - sd, mu + sd, alpha=0.15, color=color)
        ax.set_xticks(range(n_saved));  ax.set_xticklabels(labels, fontsize=8)
        ax.axhline(0.5, color='gray', ls=':', lw=0.8)
        ax.set_ylim(0.3, 1.02)
        ax.grid(alpha=0.3)
        if row == 0:
            ax.set_title(MODEL_LABELS[m])
        if col == 0:
            ax.set_ylabel(f'{task}  AUROC')
        if row == 1:
            ax.set_xlabel('layer')
        ax.legend(fontsize=8, loc='lower right', frameon=False)

fig.suptitle('Fig 3 — Token-position roles across layers (frameworks averaged)', y=1.01)
fig.tight_layout()
fig.savefig(FIG_DIR / '14_fig3_position_collapse.png')
plt.show()


## 7. Figure 4 — Context carries the signal

Among *harmful* conversations, group by `attack_success`: accepted (bypassed safety)
vs refused. For each layer, compute the fraction of refused conversations whose
final-turn representation is closer to the *accepted-harmful* centroid than to the
*benign* centroid. If the model has internalised the conversation as harmful, refused
trajectories still cluster with accepted-harmful; if only the literal message matters,
they cluster with benign.

Rows = frameworks {Crescendo, ActorAttack}. Columns = models. Lines = full-context
vs no-context — the gap is the contribution of accumulated history.

In [ ]:
def nearest_centroid_fraction(data_dict, fw, h_key):
    '''At each saved layer, fraction of refused-harmful final-turn vectors nearer
       to accepted-harmful centroid than to benign centroid. Returns (n_saved,) or None.'''
    Xh = data_dict.get((fw, 'harmful'));  Xb = data_dict.get((fw, 'benign'))
    if Xh is None or Xb is None:
        return None
    if 'attack_success' not in Xh['meta'].columns:
        return None
    idx_h = final_turn_idx(Xh['meta']);  idx_b = final_turn_idx(Xb['meta'])
    meta_h = Xh['meta'].loc[idx_h]
    y = meta_h['attack_success'].astype(bool).values
    if y.sum() < 5 or (~y).sum() < 5:
        return None
    arr_h = Xh[h_key][idx_h].astype(np.float32)
    arr_b = Xb[h_key][idx_b].astype(np.float32)
    n_saved = arr_h.shape[1]
    out = np.full(n_saved, np.nan)
    for L in range(n_saved):
        c_acc = arr_h[y,  L].mean(0)
        c_ref = arr_h[~y, L]
        c_ben = arr_b[:,  L].mean(0)
        d_acc = np.linalg.norm(c_ref - c_acc, axis=-1)
        d_ben = np.linalg.norm(c_ref - c_ben, axis=-1)
        out[L] = float((d_acc < d_ben).mean())
    return out


FWS4 = ['crescendo', 'actorattack']
fig, axes = plt.subplots(len(FWS4), max(1, len(MD)),
                         figsize=(3.8 * max(1, len(MD)), 3.2 * len(FWS4)),
                         sharey=True, squeeze=False)

for r, fw in enumerate(FWS4):
    for c, (m, d) in enumerate(MD.items()):
        ax = axes[r, c]
        n_saved = d['layer_info']['n_sweep']
        labels  = d['layer_info']['labels']

        full = nearest_centroid_fraction(d['traj'], fw, 'h_inst')
        nc   = nearest_centroid_fraction(d['nc'],   fw, 'h_inst')
        if full is not None:
            ax.plot(range(n_saved), full, 'o-', color='#1f77b4', lw=2, label='full-context')
        if nc is not None:
            ax.plot(range(n_saved), nc, 's-', color='#d62728', lw=2, label='no-context')
        ax.set_xticks(range(n_saved));  ax.set_xticklabels(labels, fontsize=8)
        ax.axhline(0.5, color='gray', ls=':', lw=0.8)
        ax.set_ylim(0.0, 1.02)
        ax.grid(alpha=0.3)
        if r == 0:
            ax.set_title(MODEL_LABELS[m])
        if c == 0:
            ax.set_ylabel(f'{fw}\nfrac(refused ≈ accepted)')
        if r == len(FWS4) - 1:
            ax.set_xlabel('layer')
        ax.legend(fontsize=8, loc='lower right', frameon=False)

fig.suptitle('Fig 4 — Where the harmfulness signal lives (nearest-centroid, h_inst, final turn)', y=1.01)
fig.tight_layout()
fig.savefig(FIG_DIR / '14_fig4_context_signal.png')
plt.show()


## 8. Summary export

Writes a compact per-model summary table to `data/cross_model_summary.csv`.

In [ ]:
rows = []
for m, d in MD.items():
    L = d['focal_pos']
    v_st = d['v_st'][H_KEY_DEFAULT]
    # displacement at final-k (collapsed across frameworks)
    disp_full, disp_nc = [], []
    for fw in FRAMEWORKS:
        vf_last = None
        for k in range(MAX_K, 0, -1):
            v = d['v_full'].get((fw, H_KEY_DEFAULT), {}).get(k)
            if v is not None:
                vf_last = v;  break
        if vf_last is not None and v_st is not None:
            disp_full.append(cosine(vf_last[L], v_st[L]))
        vnc_last = None
        for k in range(MAX_K, 0, -1):
            v = d['v_nc'].get((fw, H_KEY_DEFAULT), {}).get(k)
            if v is not None:
                vnc_last = v;  break
        if vnc_last is not None and v_st is not None:
            disp_nc.append(cosine(vnc_last[L], v_st[L]))
    rows.append({
        'model':              m,
        'focal_layer':        d['focal_label'],
        'cos_full_final':     float(np.mean(disp_full)) if disp_full else np.nan,
        'cos_nc_final':       float(np.mean(disp_nc))   if disp_nc   else np.nan,
        'fixed_ST_AUROC':     float(df2.loc[df2.model == m, 'fixed_ST'].mean())  if len(df2) else np.nan,
        'adaptive_AUROC':     float(df2.loc[df2.model == m, 'adaptive'].mean())  if len(df2) else np.nan,
        'ST_to_MLP_AUROC':    float(df2.loc[df2.model == m, 'ST_to_MLP'].mean()) if len(df2) else np.nan,
    })
summary = pd.DataFrame(rows)
out_path = repo_root / 'data' / 'cross_model_summary.csv'
summary.to_csv(out_path, index=False)
print(f'Wrote {out_path}')
summary.round(3)
